In [2]:
import pandas as pd
import duckdb as db
import requests
from io import StringIO

In [3]:
tpa = db.read_csv("../../data/TrafficPerAirport.csv")
tpt = db.read_csv("../../data/TrafficPerTerritory.csv")
territory = db.read_csv('../../data/Territory.csv')
airservice = db.read_csv('../../data/AirService.csv')
aircraftmovement = db.read_csv('../../data/AircraftMovement.csv')
airport = db.read_csv('../../data/Airport.csv')

## 1) Total observed values per Island in the last N months

[Date functions in duckdb](https://duckdb.org/docs/stable/sql/functions/date.html)

DATE_SUB vs DATE_DIFF

In [5]:
# Last year
Nmonths = 12
# 0 arrival, 1 departure
aircraftsMovementsAllowed = [0, 1]

In [6]:
listbetparenthesis = "("
for a in aircraftsMovementsAllowed: 
    listbetparenthesis += f"{a},"
listbetparenthesis = listbetparenthesis[0:-1] # Delete last comma
listbetparenthesis += ")"

In [7]:
listbetparenthesis

'(0,1)'

In [8]:
if Nmonths == "total":
    Nmonths = db.sql("SELECT DATE_SUB('month', MIN(Month), MAX(Month)) FROM tpt").fetchone()[0]

In [9]:

db.sql( f"\
        SELECT T.TerritoryName, SUM(Passengers) AS Total_Passengers, SUM(Operations) AS Total_Op, SUM(Goods) AS Total_Goods, SUM(Mail) AS Total_Mail \
        FROM tpt INNER JOIN territory T ON T.TerritoryId = tpt.IslandId \
        WHERE tpt.AircraftMovementId IN {listbetparenthesis} AND T.TerritoryName != 'Canary Islands' \
        AND DATE_SUB('month', Month, (SELECT MAX(month) FROM tpt)) <= {Nmonths}  \
        GROUP BY T.TerritoryName \
        ORDER BY Total_Passengers DESC")

┌───────────────┬──────────────────┬──────────┬─────────────┬────────────┐
│ TerritoryName │ Total_Passengers │ Total_Op │ Total_Goods │ Total_Mail │
│    varchar    │      int128      │  int128  │   int128    │   int128   │
├───────────────┼──────────────────┼──────────┼─────────────┼────────────┤
│ Tenerife      │         61069064 │   454848 │    21168118 │    7191922 │
│ Gran Canaria  │         42089972 │   350968 │    41297158 │    3065838 │
│ Lanzarote     │         27773260 │   205714 │     1099578 │       1064 │
│ Fuerteventura │         21586922 │   157154 │      785904 │          0 │
│ La Palma      │          3645070 │    52818 │      632312 │         32 │
│ El Hierro     │           698782 │    13390 │      130458 │          8 │
│ La Gomera     │           274194 │     6292 │        7878 │          8 │
└───────────────┴──────────────────┴──────────┴─────────────┴────────────┘

## 2) Total observed values in TrafficPerAirport filtered by BaseAirport, StopoverAirport, airservice, aricraftmovement  in the between N and M dates

In [10]:
s_date = '2024-01-01'
end_date = '2025-07-01'

In [11]:
# 0 arrival, 1 departure
aircraftmov = [0, 1]
# 0 Commercial (total), 1 Other, 2 Non scheduled, 3 regular
airservice = [0]

In [12]:
def list_to_string(list): 
    st = "("
    for a in list: 
        st += f"{a},"
    st = st[0:-1] # Delete last comma
    st += ")"
    return st

In [13]:
aircraftmovstr = list_to_string(aircraftmov)

airservicestr = list_to_string(airservice)

In [14]:


db.sql( f"\
        SELECT BaseAirportId, StopoverAirportId, SUM(Passengers) AS Total_Passengers, SUM(Operations) AS Total_Op, SUM(Goods) AS Total_Goods, SUM(Mail) AS Total_Mail \
        FROM tpa \
        WHERE AircraftMovementId IN {aircraftmovstr} AND AirServiceId IN {airservicestr}\
        AND Month BETWEEN '{s_date}' AND '{end_date
                                           }'  \
        GROUP BY BaseAirportId, StopoverAirportId \
        ORDER BY Total_Passengers DESC")

┌───────────────┬───────────────────┬──────────────────┬──────────┬─────────────┬────────────┐
│ BaseAirportId │ StopoverAirportId │ Total_Passengers │ Total_Op │ Total_Goods │ Total_Mail │
│     int64     │       int64       │      int128      │  int128  │   int128    │   int128   │
├───────────────┼───────────────────┼──────────────────┼──────────┼─────────────┼────────────┤
│            47 │               111 │          3091032 │    20857 │    19502044 │     576386 │
│           137 │               111 │          2815571 │    18819 │     8369751 │    3222134 │
│            47 │               137 │          1562642 │    28262 │     2999526 │    1626394 │
│           137 │                47 │          1559923 │    28270 │     5145590 │    2104291 │
│           165 │                43 │          1524349 │     7780 │       46991 │          0 │
│            47 │                54 │          1346820 │    24331 │      362237 │        242 │
│            54 │                47 │          134

## 3) Stopover airports where more passengers arrive than leave per base airport

In [15]:
db.sql('WITH arrival_departure AS ( SELECT BaseAirportId, StopoverAirportId,\
       SUM(Passengers) FILTER (WHERE AircraftMovementId = 0) AS ArrivalPassengers,\
       SUM(Passengers) FILTER (WHERE AircraftMovementId = 1) AS DeparturePassengers\
       FROM tpa\
       GROUP BY BaseAirportId, StopoverAirportId\
       )\
        \
       SELECT AirportName, COUNT(StopoverAirportId) AS Number_of_airports_where_departure_greater_than_arrival\
       FROM arrival_departure ad INNER JOIN airport a ON ad.BaseAirportId = a.AirportId\
       WHERE ArrivalPassengers < DeparturePassengers\
       GROUP BY AirportName\
       ORDER BY Number_of_airports_where_departure_greater_than_arrival DESC')

┌────────────────────────┬─────────────────────────────────────────────────────────┐
│      AirportName       │ Number_of_airports_where_departure_greater_than_arrival │
│        varchar         │                          int64                          │
├────────────────────────┼─────────────────────────────────────────────────────────┤
│ Gran Canaria Airport   │                                                     132 │
│ Tenerife Norte Airport │                                                     127 │
│ Lanzarote Airport      │                                                     101 │
│ Fuerteventura Airport  │                                                      98 │
│ Tenerife South Airport │                                                      97 │
│ La Palma Airport       │                                                      67 │
│ La Gomera Airport      │                                                       8 │
│ Hierro Airport         │                                       

In [16]:
db.sql('WITH arrival_departure AS ( SELECT BaseAirportId, StopoverAirportId,\
       SUM(Passengers) FILTER (WHERE AircraftMovementId = 0) AS ArrivalPassengers,\
       SUM(Passengers) FILTER (WHERE AircraftMovementId = 1) AS DeparturePassengers\
       FROM tpa\
       GROUP BY BaseAirportId, StopoverAirportId\
       )\
        \
       SELECT a.AirportName AS Base, a2.AirportName AS Stopover\
       FROM arrival_departure ad INNER JOIN airport a ON ad.BaseAirportId = a.AirportId INNER JOIN airport a2 ON ad.StopoverAirportId = a2.AirportId\
       WHERE ArrivalPassengers < DeparturePassengers\
       \
       ORDER BY Base')

┌────────────────────────┬───────────────────────────────────────┐
│          Base          │               Stopover                │
│        varchar         │                varchar                │
├────────────────────────┼───────────────────────────────────────┤
│ Fuerteventura Airport  │ Paderborn Lippstadt Airport           │
│ Fuerteventura Airport  │ Adolfo Suárez Madrid-Barajas Airport  │
│ Fuerteventura Airport  │ Vnukovo International Airport         │
│ Fuerteventura Airport  │ Deauville-Saint-Gatien Airport        │
│ Fuerteventura Airport  │ Aberdeen Dyce Airport                 │
│ Fuerteventura Airport  │ Kuopio Airport                        │
│ Fuerteventura Airport  │ Saarbrücken Airport                   │
│ Fuerteventura Airport  │ Kalmar Airport                        │
│ Fuerteventura Airport  │ Norwich International Airport         │
│ Fuerteventura Airport  │ Barcelona International Airport       │
│           ·            │           ·                        

## 4) Share of arrivals vs departures passengers

In [17]:
db.sql('WITH t AS (SELECT IslandId, StopoverTerritoryId,\
       SUM(Passengers) FILTER (WHERE AircraftMovementId = 0) AS ArrivalPassengers,\
       SUM(Passengers) FILTER (WHERE AircraftMovementId = 1) AS DeparturePassengers, SUM(Passengers) FILTER (WHERE AircraftMovementId = 2) AS total\
       FROM tpt WHERE AirServiceId = 0\
       GROUP BY IslandId, StopoverTerritoryId)\
       \
       SELECT te.TerritoryName AS Island, te2.TerritoryName AS Stopover,\
       COALESCE(ROUND((ArrivalPassengers/NULLIF(total, 0))*100, 2), 0) AS ArrivalPer,\
       COALESCE(ROUND((DeparturePassengers/NULLIF(total, 0))*100, 2), 0) AS DeparturePer,\
       ArrivalPassengers, DeparturePassengers\
       FROM t INNER JOIN territory te ON t.IslandId = te.TerritoryId INNER JOIN territory te2 ON te2.TerritoryId = t.StopoverTerritoryId\
       ORDER BY DeparturePer DESC')

┌────────────────┬──────────────────────────────────────────────────────┬────────────┬──────────────┬───────────────────┬─────────────────────┐
│     Island     │                       Stopover                       │ ArrivalPer │ DeparturePer │ ArrivalPassengers │ DeparturePassengers │
│    varchar     │                       varchar                        │   double   │    double    │      int128       │       int128        │
├────────────────┼──────────────────────────────────────────────────────┼────────────┼──────────────┼───────────────────┼─────────────────────┤
│ El Hierro      │ Spain (Canary Islands excluded)                      │      22.78 │        77.22 │                36 │                 122 │
│ La Gomera      │ Foreign                                              │      33.33 │        66.67 │                18 │                  36 │
│ La Gomera      │ Spain (Canary Islands excluded)                      │      46.43 │        53.57 │                26 │               

## 5) Top 3 busiest airports per year

In [18]:
db.sql('WITH totalPerYear AS (SELECT YEAR(Month) AS year, AirportName, SUM(Passengers) AS t_Pass \
        FROM tpa INNER JOIN airport ON BaseAirportId = AirportId\
        WHERE AircraftMovementId = 2 AND AirServiceId = 0\
        GROUP BY 1,2)\
        \
        SELECT * FROM\
        (SELECT year,AirportName, DENSE_RANK() OVER (PARTITION BY year ORDER BY t_Pass DESC) AS rank FROM totalPerYear)\
        WHERE rank <= 3 ORDER BY year DESC, rank ASC')

┌───────┬────────────────────────┬───────┐
│ year  │      AirportName       │ rank  │
│ int64 │        varchar         │ int64 │
├───────┼────────────────────────┼───────┤
│  2025 │ Gran Canaria Airport   │     1 │
│  2025 │ Tenerife South Airport │     2 │
│  2025 │ Lanzarote Airport      │     3 │
│  2024 │ Gran Canaria Airport   │     1 │
│  2024 │ Tenerife South Airport │     2 │
│  2024 │ Lanzarote Airport      │     3 │
│  2023 │ Gran Canaria Airport   │     1 │
│  2023 │ Tenerife South Airport │     2 │
│  2023 │ Lanzarote Airport      │     3 │
│  2022 │ Gran Canaria Airport   │     1 │
│    ·  │         ·              │     · │
│    ·  │         ·              │     · │
│    ·  │         ·              │     · │
│  2007 │ Lanzarote Airport      │     3 │
│  2006 │ Gran Canaria Airport   │     1 │
│  2006 │ Tenerife South Airport │     2 │
│  2006 │ Lanzarote Airport      │     3 │
│  2005 │ Gran Canaria Airport   │     1 │
│  2005 │ Tenerife South Airport │     2 │
│  2005 │ L

## 6) Monthly Passenger Growth Rate by Island (YoY % Change)

In [20]:
db.sql('WITH grouped AS (SELECT YEAR(Month) AS year, IslandId, SUM(Passengers) AS Passengers\
        FROM tpt WHERE AircraftMovementId = 2 AND AirServiceId = 0 GROUP BY 1, 2)\
        \
        SELECT year, TerritoryName, ROUND(((Passengers - lag)/lag)*100, 2) AS YoY_Percentage_Change, Passengers, lag\
        FROM (SELECT *, LAG(Passengers, 1) OVER (PARTITION BY IslandId ORDER BY year) AS lag\
              FROM grouped\
              ) INNER JOIN territory ON IslandId = TerritoryId\
        ORDER BY TerritoryName, year\
        ')

┌───────┬────────────────┬───────────────────────┬────────────┬──────────┐
│ year  │ TerritoryName  │ YoY_Percentage_Change │ Passengers │   lag    │
│ int64 │    varchar     │        double         │   int128   │  int128  │
├───────┼────────────────┼───────────────────────┼────────────┼──────────┤
│  2004 │ Canary Islands │                  NULL │   44211733 │     NULL │
│  2005 │ Canary Islands │                  1.21 │   44745800 │ 44211733 │
│  2006 │ Canary Islands │                  3.41 │   46272964 │ 44745800 │
│  2007 │ Canary Islands │                 -0.51 │   46038195 │ 46272964 │
│  2008 │ Canary Islands │                 -2.04 │   45097355 │ 46038195 │
│  2009 │ Canary Islands │                 -12.3 │   39551669 │ 45097355 │
│  2010 │ Canary Islands │                  5.31 │   41653203 │ 39551669 │
│  2011 │ Canary Islands │                 13.41 │   47237398 │ 41653203 │
│  2012 │ Canary Islands │                 -5.46 │   44658973 │ 47237398 │
│  2013 │ Canary Islands 

All islands

In [21]:
db.sql('WITH grouped AS (SELECT YEAR(Month) AS year, IslandId, SUM(Passengers) AS Passengers\
        FROM tpt WHERE AircraftMovementId = 2 AND AirServiceId = 0 GROUP BY 1, 2)\
        \
        SELECT year, IslandId, ROUND(((Passengers - lag)/lag)*100, 2) AS YoY_Percentage_Change, Passengers, lag\
        FROM (SELECT *, LAG(Passengers, 1) OVER (PARTITION BY IslandId ORDER BY year) AS lag\
              FROM grouped\
              ) WHERE IslandId = 0\
        ORDER BY IslandId, year\
        ')

┌───────┬──────────┬───────────────────────┬────────────┬──────────┐
│ year  │ IslandId │ YoY_Percentage_Change │ Passengers │   lag    │
│ int64 │  int64   │        double         │   int128   │  int128  │
├───────┼──────────┼───────────────────────┼────────────┼──────────┤
│  2004 │        0 │                  NULL │   44211733 │     NULL │
│  2005 │        0 │                  1.21 │   44745800 │ 44211733 │
│  2006 │        0 │                  3.41 │   46272964 │ 44745800 │
│  2007 │        0 │                 -0.51 │   46038195 │ 46272964 │
│  2008 │        0 │                 -2.04 │   45097355 │ 46038195 │
│  2009 │        0 │                 -12.3 │   39551669 │ 45097355 │
│  2010 │        0 │                  5.31 │   41653203 │ 39551669 │
│  2011 │        0 │                 13.41 │   47237398 │ 41653203 │
│  2012 │        0 │                 -5.46 │   44658973 │ 47237398 │
│  2013 │        0 │                 -0.21 │   44565038 │ 44658973 │
│  2014 │        0 │              

## 7) Top 3 Months with Highest Passenger Volume per Territory

In [55]:
db.sql("WITH gr AS\
(SELECT MONTH(t.Month) AS Month, t.IslandId,\
ROUND(AVG(t.Passengers))::Integer AS Pass \
FROM tpt t WHERE t.AirServiceId = 0 AND t.AircraftMovementId = 2 GROUP BY 1, 2)\
\
SELECT TerritoryName, Month, rank, Pass FROM \
(SELECT *, DENSE_RANK() OVER (PARTITION BY IslandId ORDER BY Pass DESC) AS rank FROM gr) t INNER JOIN territory te ON t.IslandId = te.TerritoryId \
WHERE rank <= 3 ORDER BY 1,3")

┌────────────────┬───────┬───────┬────────┐
│ TerritoryName  │ Month │ rank  │  Pass  │
│    varchar     │ int64 │ int64 │ int32  │
├────────────────┼───────┼───────┼────────┤
│ Canary Islands │     3 │     1 │ 909358 │
│ Canary Islands │     8 │     2 │ 879862 │
│ Canary Islands │    10 │     3 │ 876106 │
│ El Hierro      │     8 │     1 │   4264 │
│ El Hierro      │     7 │     2 │   3989 │
│ El Hierro      │     9 │     3 │   3635 │
│ Fuerteventura  │     8 │     1 │ 136984 │
│ Fuerteventura  │    10 │     2 │ 130895 │
│ Fuerteventura  │     7 │     3 │ 128155 │
│ Gran Canaria   │     3 │     1 │ 258418 │
│     ·          │     · │     · │     ·  │
│     ·          │     · │     · │     ·  │
│     ·          │     · │     · │     ·  │
│ La Gomera      │     9 │     3 │   1027 │
│ La Palma       │     3 │     1 │  23818 │
│ La Palma       │     8 │     2 │  23779 │
│ La Palma       │     7 │     3 │  22632 │
│ Lanzarote      │     8 │     1 │ 165200 │
│ Lanzarote      │     7 │     2

## 8) Percentage of passengers that arrive in the Canary Islands by Country (Market share per country)

In the case of Spain we will only count airports outside the canary islands (all spain canary island excluded)

In [98]:
db.sql("SELECT a1.CountryName,  \
 ROUND(SUM(Passengers)/ \
            (SELECT SUM(Passengers) FROM tpa ta INNER JOIN airport a2 ON ta.StopoverAirportId = a2.AirportId \
                WHERE ta.AirServiceId = 0 AND ta.AircraftMovementId = 2 AND a2.AirportCode NOT LIKE 'ES_GC%') * 100, 2) AS per  \
FROM tpa INNER JOIN airport a1 ON StopoverAirportId = a1.AirportId \
WHERE AirServiceId = 0 AND AircraftMovementId = 2 AND a1.AirportCode NOT LIKE 'ES_GC%'\
GROUP BY a1.CountryName ORDER BY 2 DESC\
")

┌──────────────────────────────────────────────────────┬────────┐
│                     CountryName                      │  per   │
│                       varchar                        │ double │
├──────────────────────────────────────────────────────┼────────┤
│ United Kingdom of Great Britain and Northern Ireland │  27.96 │
│ Spain                                                │  25.77 │
│ Germany                                              │  17.13 │
│ Netherlands                                          │   3.26 │
│ Ireland                                              │   3.19 │
│ Denmark                                              │   2.73 │
│ Sweden                                               │   2.71 │
│ France                                               │   2.59 │
│ Norway                                               │   2.56 │
│ Italy                                                │   2.43 │
│   ·                                                  │     ·  │
│   ·     